# 02. 데이터 품질 + 텍스트 분석

**목표**: 데이터 품질 이슈 발견 + 텍스트 길이 분포로 청킹 전략 도출

**의존**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

**산출물**: `eda_output/phase3_quality.json`, `eda_output/phase4_text.json`

In [ ]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import re
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.auto import tqdm

from scripts.eda.common import (
    DATA_DIR,
    get_sample,
    load_result,
    save_result,
)
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

# 이전 단계 결과 로드
phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")
print(f"Phase 1 로드: {len(phase1)}개 파일")
print(f"Phase 2 로드: {len(phase2)}개 카테고리 스키마")

# ── 샘플 캐시 (대용량 파일 반복 로딩 방지) ──────────────
# fast=True: head sampling (처음 N개만 읽고 중단, 전체 파일 스캔 없음)
SAMPLE_SIZE = 5000
_sample_cache: dict[str, list[dict]] = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="샘플 로딩"):
    first_file = cat_info["files"][0]
    filepath = DATA_DIR / first_file
    if not filepath.exists():
        continue
    _sample_cache[cat_key] = get_sample(filepath, n=SAMPLE_SIZE, fast=True)
    print(f"  {cat_info['label']}: {len(_sample_cache[cat_key]):,}건")

print(f"\n총 {len(_sample_cache)}개 카테고리 캐시 완료")

## 1. 데이터 품질 분석

카테고리별 5,000건 샘플에서 null/empty 비율, 중복 ID, 인코딩 이슈를 점검합니다.

In [ ]:
# ── 카테고리별 품질 분석 ──────────────────────────────────
quality_results = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="품질 분석"):
    if cat_key not in _sample_cache:
        continue

    sample = _sample_cache[cat_key]
    total = len(sample)
    if total == 0:
        continue

    # 필드별 null/empty 분석
    field_quality: dict[str, dict] = {}
    for record in sample:
        for key, value in record.items():
            if key not in field_quality:
                field_quality[key] = {"null": 0, "empty": 0, "present": 0, "null_str": 0}
            fq = field_quality[key]
            fq["present"] += 1
            if value is None:
                fq["null"] += 1
            elif isinstance(value, str):
                if value.strip() == "":
                    fq["empty"] += 1
                elif value.strip().lower() == "null":
                    fq["null_str"] += 1

    # ID 중복 분석
    id_field = cat_info.get("id_field")
    duplicate_ids = 0
    duplicate_rate = 0.0
    if id_field:
        # ID 값이 list인 경우 str로 변환하여 hashable하게 처리
        ids = []
        for r in sample:
            v = r.get(id_field)
            if v is None:
                continue
            ids.append(str(v) if isinstance(v, list) else v)
        id_counts = Counter(ids)
        duplicate_ids = sum(1 for c in id_counts.values() if c > 1)
        duplicate_rate = duplicate_ids / len(id_counts) if id_counts else 0

    # 인코딩 이슈 탐지
    encoding_issues = 0
    html_entities = 0
    for record in sample[:1000]:  # 1000건만 체크
        for value in record.values():
            if not isinstance(value, str):
                continue
            if "\ufffd" in value or "\\u" in value:
                encoding_issues += 1
            if "&amp;" in value or "&lt;" in value or "&gt;" in value:
                html_entities += 1

    quality_results[cat_key] = {
        "label": cat_info["label"],
        "sample_count": total,
        "field_quality": {
            k: {
                "null_rate": round(v["null"] / v["present"], 4) if v["present"] > 0 else 0,
                "empty_rate": round(v["empty"] / v["present"], 4) if v["present"] > 0 else 0,
                "null_str_count": v["null_str"],
            }
            for k, v in field_quality.items()
        },
        "duplicate_id_field": id_field,
        "duplicate_ids": duplicate_ids,
        "duplicate_rate": round(duplicate_rate, 4),
        "encoding_issues": encoding_issues,
        "html_entities": html_entities,
    }
    print(f"  {cat_info['label']}: dup={duplicate_rate:.2%}, encoding={encoding_issues}, html={html_entities}")

In [ ]:
# ── null률 히트맵 (카테고리 x 필드) ──────────────────────
# 주요 필드만 선택 (각 카테고리의 text_fields + id_field + date_field)
key_fields_per_cat = {}
for cat_key, cat_info in CATEGORIES.items():
    fields = set()
    if cat_info.get("id_field"):
        fields.add(cat_info["id_field"])
    if cat_info.get("date_field"):
        fields.add(cat_info["date_field"])
    for tf in cat_info.get("text_fields", []):
        fields.add(tf)
    key_fields_per_cat[cat_key] = fields

# 모든 주요 필드의 합집합
all_key_fields = sorted(set().union(*key_fields_per_cat.values()))

# 히트맵 데이터 구성
heatmap_data = []
cat_labels = []
for cat_key, qr in quality_results.items():
    cat_labels.append(qr["label"])
    row = []
    for field in all_key_fields:
        fq = qr["field_quality"].get(field)
        if fq is None:
            row.append(None)  # 해당 카테고리에 이 필드 없음
        else:
            row.append(fq["null_rate"] + fq["empty_rate"])  # null + empty 합산
    heatmap_data.append(row)

fig = px.imshow(
    heatmap_data,
    x=all_key_fields,
    y=cat_labels,
    title="카테고리 x 주요 필드 null/empty 비율 히트맵",
    labels=dict(x="필드", y="카테고리", color="null+empty 비율"),
    color_continuous_scale="YlOrRd",
    aspect="auto",
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# ── 중복 ID 비율 바 차트 ─────────────────────────────────
dup_data = [
    {
        "카테고리": qr["label"],
        "중복 비율": qr["duplicate_rate"],
        "중복 ID 수": qr["duplicate_ids"],
        "ID 필드": qr["duplicate_id_field"] or "없음",
    }
    for qr in quality_results.values()
]
df_dup = pd.DataFrame(dup_data).sort_values("중복 비율", ascending=True)

fig = px.bar(
    df_dup,
    x="중복 비율",
    y="카테고리",
    orientation="h",
    title="카테고리별 ID 중복 비율",
    hover_data=["중복 ID 수", "ID 필드"],
    color="중복 비율",
    color_continuous_scale="Reds",
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 2. 텍스트 길이 분석

주요 텍스트 필드의 길이 분포를 분석하여 청킹 전략을 도출합니다.

In [ ]:
# ── 카테고리별 텍스트 길이 수집 ──────────────────────────
text_stats = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="텍스트 분석"):
    text_fields = cat_info.get("text_fields", [])
    if not text_fields:
        continue

    if cat_key not in _sample_cache:
        continue

    sample = _sample_cache[cat_key]

    field_lengths: dict[str, list[int]] = {f: [] for f in text_fields}

    for record in sample:
        for field in text_fields:
            value = record.get(field)
            if isinstance(value, str) and value.strip():
                field_lengths[field].append(len(value))
            elif isinstance(value, list):
                # 배열 필드 (조문내용 등): 전체 직렬화 길이
                text = str(value)
                field_lengths[field].append(len(text))

    cat_stats = {}
    for field, lengths in field_lengths.items():
        if not lengths:
            continue
        arr = np.array(lengths)
        cat_stats[field] = {
            "count": len(lengths),
            "min": int(arr.min()),
            "p25": int(np.percentile(arr, 25)),
            "p50": int(np.percentile(arr, 50)),
            "p75": int(np.percentile(arr, 75)),
            "p90": int(np.percentile(arr, 90)),
            "p99": int(np.percentile(arr, 99)),
            "max": int(arr.max()),
            "mean": round(float(arr.mean()), 1),
        }

    text_stats[cat_key] = {
        "label": cat_info["label"],
        "fields": cat_stats,
    }
    for field, stats in cat_stats.items():
        print(f"  {cat_info['label']}.{field}: P50={stats['p50']:,}, P90={stats['p90']:,}, max={stats['max']:,}")

In [ ]:
# ── 카테고리별 텍스트 길이 box plot ──────────────────────
# 주요 텍스트 필드의 P25~P99 범위를 box plot으로 시각화
box_data = []
for cat_key, ts in text_stats.items():
    for field, stats in ts["fields"].items():
        # box plot용 5-number summary 생성
        box_data.append({
            "카테고리": ts["label"],
            "필드": field,
            "레이블": f"{ts['label']}\n{field}",
            "P25": stats["p25"],
            "P50": stats["p50"],
            "P75": stats["p75"],
            "P90": stats["p90"],
            "P99": stats["p99"],
            "mean": stats["mean"],
        })

df_box = pd.DataFrame(box_data)

fig = go.Figure()
for _, row in df_box.iterrows():
    fig.add_trace(go.Box(
        name=row["레이블"],
        q1=[row["P25"]],
        median=[row["P50"]],
        q3=[row["P75"]],
        lowerfence=[row["P25"]],  # P25를 하한으로 사용
        upperfence=[row["P90"]],
        mean=[row["mean"]],
        boxmean=True,
    ))

fig.update_layout(
    title="카테고리별 텍스트 길이 분포 (P25-P90, 문자 수)",
    yaxis_title="텍스트 길이 (문자)",
    height=600,
    showlegend=False,
)
fig.show()

In [ ]:
# ── 텍스트 길이 히스토그램 (카테고리 선택) ────────────────
# 대표 카테고리 선택: precedent, law, constitutional, administration
CHUNK_SIZE = 1250  # 현재 청킹 기준선

for cat_key in ["precedent", "law", "constitutional", "administration"]:
    if cat_key not in text_stats:
        continue
    ts = text_stats[cat_key]
    # 첫 번째 텍스트 필드의 길이 분포
    first_field = list(ts["fields"].keys())[0]
    stats = ts["fields"][first_field]

    # 캐시된 샘플에서 길이 데이터 수집
    if cat_key not in _sample_cache:
        continue
    sample = _sample_cache[cat_key]
    lengths = [
        len(r.get(first_field, ""))
        for r in sample
        if isinstance(r.get(first_field), str) and r.get(first_field, "").strip()
    ]

    if not lengths:
        continue

    fig = px.histogram(
        x=lengths,
        nbins=100,
        title=f"{ts['label']} - {first_field} 텍스트 길이 분포",
        labels={"x": "문자 수", "y": "빈도"},
    )
    fig.add_vline(
        x=CHUNK_SIZE,
        line_dash="dash",
        line_color="red",
        annotation_text=f"청킹 기준 ({CHUNK_SIZE}자)",
    )
    fig.update_layout(height=400)
    fig.show()

In [ ]:
# ── 예상 청크 수 분석 ────────────────────────────────────
# 현재 청킹 설정: 1,250자 / 800토큰 기준
CHUNK_CHAR_LIMIT = 1250
OVERLAP_CHARS = 200  # 가정: 200자 오버랩

chunking_analysis = []
for cat_key, ts in text_stats.items():
    for field, stats in ts["fields"].items():
        # 평균 텍스트 길이 기준 예상 청크 수
        effective_chunk = CHUNK_CHAR_LIMIT - OVERLAP_CHARS
        avg_chunks = max(1, stats["mean"] / effective_chunk) if effective_chunk > 0 else 1
        p90_chunks = max(1, stats["p90"] / effective_chunk) if effective_chunk > 0 else 1

        chunking_analysis.append({
            "카테고리": ts["label"],
            "필드": field,
            "평균 길이": stats["mean"],
            "P90 길이": stats["p90"],
            "평균 청크 수": round(avg_chunks, 1),
            "P90 청크 수": round(p90_chunks, 1),
            "청킹 필요": "예" if stats["p50"] > CHUNK_CHAR_LIMIT else "아니오",
        })

df_chunk = pd.DataFrame(chunking_analysis)

# 청킹 전략 권고 테이블
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_chunk.columns),
        fill_color="#548235",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_chunk[col] for col in df_chunk.columns],
        fill_color="#F2F2F2",
        align="left",
        font=dict(size=11),
        height=28,
    ),
)])
fig.update_layout(
    title=f"청킹 전략 권고 (기준: {CHUNK_CHAR_LIMIT}자, 오버랩: {OVERLAP_CHARS}자)",
    height=max(400, len(chunking_analysis) * 30 + 100),
)
fig.show()

In [ ]:
# ── 결과 저장 ─────────────────────────────────────────────
p3_path = save_result("phase3_quality", quality_results)
print(f"Phase 3 저장: {p3_path}")

p4_path = save_result("phase4_text", text_stats)
print(f"Phase 4 저장: {p4_path}")

# 요약 출력
print(f"\n=== 품질 요약 ===")
for cat_key, qr in quality_results.items():
    issues = []
    if qr["duplicate_rate"] > 0:
        issues.append(f"ID중복 {qr['duplicate_rate']:.1%}")
    if qr["encoding_issues"] > 0:
        issues.append(f"인코딩 {qr['encoding_issues']}건")
    if qr["html_entities"] > 0:
        issues.append(f"HTML {qr['html_entities']}건")
    issue_str = ", ".join(issues) if issues else "이슈 없음"
    print(f"  {qr['label']}: {issue_str}")